In [23]:
import pandas as pd
import csv

In [24]:
def parse_params(params_str):
    params = {}
    for item in params_str.split():
        if '=' in item:
            key, value = item.split('=', 1)
            params[key] = value
    return params

# aggregates results over all parameters except those in params_to_compare
def results_csv_to_df(csv_file, params_to_aggregate=None):
    results = []

    with open(csv_file, newline='') as f:
        reader = csv.DictReader(f, fieldnames=[
            'accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed', 'not_fully_unmasked',
            'checkpoint', 'strategy', 'params'
        ])
        reader.__next__()  # Skip header row
        for row in reader:
            params = parse_params(row['params'])
            row_dict = {
                'accuracy': float(row['accuracy']),
                'correctly_filled_cells': float(row['correctly_filled_cells']),
                'checkpoint': row['checkpoint'],
                'strategy': row['strategy'],
                'nfe': float(row['nfe']),
                'time': float(row['time']),
                'speed': float(row['speed']),
            }
            row_dict.update(params.items())
            results.append(row_dict)
    
    results_df = pd.DataFrame(results)
    print(f"Total rows before aggregation: {len(results_df)}")
    
    # Aggregate the accuracy, correctly_filled_cells, nfe, time, and speed over the rows where all parameters except those in params_to_aggregate are the same
    if params_to_aggregate is not None:
        groupby_cols = [col for col in results_df.columns if col not in params_to_aggregate + ['accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed']]
    else:
        groupby_cols = [col for col in results_df.columns if col not in ['accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed']]
    
    aggregated_df = results_df.groupby(groupby_cols, dropna=False).agg(
        accuracy_mean=pd.NamedAgg(column='accuracy', aggfunc='mean'),
        accuracy_std=pd.NamedAgg(column='accuracy', aggfunc='std'),
        correctly_filled_cells_mean=pd.NamedAgg(column='correctly_filled_cells', aggfunc='mean'),
        correctly_filled_cells_std=pd.NamedAgg(column='correctly_filled_cells', aggfunc='std'),
        nfe=pd.NamedAgg(column='nfe', aggfunc='mean'),
        time=pd.NamedAgg(column='time', aggfunc='mean'),
        speed=pd.NamedAgg(column='speed', aggfunc='mean'),
    ).reset_index()

    print(f"Rows after aggregation: {len(aggregated_df)}")
    return aggregated_df

In [25]:
params_to_aggregate = ['seed']
df = results_csv_to_df('results_steps_before_pruning_different_strategies.csv', params_to_aggregate=params_to_aggregate)
df

Total rows before aggregation: 64
Rows after aggregation: 64


,checkpoint,strategy,score_mask_position,select_position,unmask_token,k,gumbel_noise_coefficient,dataset,num_samples,num_denoising_steps,...,position_sampling,position_metric,token_sampling,accuracy_mean,accuracy_std,correctly_filled_cells_mean,correctly_filled_cells_std,nfe,time,speed
0,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,640,80,...,NaN,NaN,NaN,0.7172,NaN,0.8744,NaN,82.16,723.0,0.88
1,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,640,80,...,NaN,NaN,NaN,0.7234,NaN,0.8766,NaN,87.80,761.0,0.84
2,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,640,80,...,NaN,NaN,NaN,0.7484,NaN,0.8882,NaN,154.68,1371.0,0.47
3,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,640,80,...,NaN,NaN,NaN,0.7500,NaN,0.8868,NaN,170.06,1355.0,0.47
4,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,640,80,...,NaN,NaN,NaN,0.6953,NaN,0.8649,NaN,150.08,1253.0,0.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,hard,640,80,...,NaN,NaN,NaN,0.6547,NaN,0.8518,NaN,582.69,4409.0,0.15
60,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,hard,640,80,...,NaN,NaN,NaN,0.6906,NaN,0.8734,NaN,85.93,674.0,0.95
61,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,hard,640,80,...,NaN,NaN,NaN,0.7016,NaN,0.8782,NaN,102.44,800.0,0.80
62,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,hard,640,80,...,NaN,NaN,NaN,0.6984,NaN,0.8765,NaN,230.48,1793.0,0.36


In [26]:
# Convert pruning_num_beams, branching_factor, steps_before_pruning to numeric for proper sorting
df['pruning_num_beams'] = pd.to_numeric(df['pruning_num_beams'], errors='coerce')
df['branching_factor'] = pd.to_numeric(df['branching_factor'], errors='coerce')
df['steps_before_pruning'] = pd.to_numeric(df['steps_before_pruning'], errors='coerce')

In [27]:
grouped_df = df.groupby(['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor', 'steps_before_pruning'], dropna=False)['accuracy_mean'].mean().unstack()#.sort_values(by='accuracy_mean')
grouped_df

steps_before_pruning                                                                                                    2   \
checkpoint                      strategy                                        pruning_num_beams branching_factor           
checkpoints/gidd_0_2/300_epochs gidd_change_based_on_model_confidence_to_change 1                 2                 0.6953   
                                                                                2                 2                 0.7156   
                                                                                                  4                 0.7219   
                                                                                4                 4                 0.7422   
                                gidd_change_low_confidence_positions            1                 2                 0.6969   
                                                                                2                 2                 0.7234   
                                                                                                  4                 0.7234   
                                                                                4                 4                 0.7422   
                                gidd_prob_to_recover_data                       1                 2                 0.7000   
                                                                                2                 2                 0.7172   
                                                                                                  4                 0.7250   
                                                                                4                 4                 0.7484   
checkpoints/mdlm/300_epochs     mdlm_adaptive_score_select_update               1                 2                 0.6875   
                                                                                2                 2                 0.6813   
                                                                                                  4                 0.6797   
                                                                                4                 4                 0.6766   

steps_before_pruning                                                                                                    4   \
checkpoint                      strategy                                        pruning_num_beams branching_factor           
checkpoints/gidd_0_2/300_epochs gidd_change_based_on_model_confidence_to_change 1                 2                 0.7078   
                                                                                2                 2                 0.7328   
                                                                                                  4                 0.7500   
                                                                                4                 4                 0.8125   
                                gidd_change_low_confidence_positions            1                 2                 0.7109   
                                                                                2                 2                 0.7422   
                                                                                                  4                 0.7578   
                                                                                4                 4                 0.8234   
                                gidd_prob_to_recover_data                       1                 2                 0.7047   
                                                                                2                 2                 0.7359   
                                                                                                  4                 0.7484   
                                                                                4            

In [28]:
# Accuracy, nfe, speed
df = df.sort_values(by=['strategy', 'pruning_num_beams', 'branching_factor', 'steps_before_pruning'])
df[['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor', 'steps_before_pruning', 'accuracy_mean', 'nfe', 'speed']]

,checkpoint,strategy,pruning_num_beams,branching_factor,steps_before_pruning,accuracy_mean,nfe,speed
4,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,1,2,2,0.6953,150.08,0.51
8,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,1,2,4,0.7078,112.83,0.65
12,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,1,2,8,0.7078,93.75,0.79
0,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,1,2,16,0.7172,82.16,0.88
5,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,2,2,2,0.7156,224.56,0.34
...,...,...,...,...,...,...,...,...
50,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,2,4,16,0.7016,150.92,0.53
55,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,4,4,2,0.6766,1052.57,0.08
59,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,4,4,4,0.6547,582.69,0.15
63,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,4,4,8,0.7000,315.35,0.27


In [37]:
# Display the values for the best steps_before_pruning for each combination of the other parameters
best_steps = df.loc[df.groupby(['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor'])['accuracy_mean'].idxmax()]
df_without_best = df[~df.index.isin(best_steps.index)]
second_best = df_without_best.loc[df_without_best.groupby(['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor'])['accuracy_mean'].idxmax()]
worst = df.loc[df.groupby(['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor'])['accuracy_mean'].idxmin()]
best_steps_margin = best_steps.copy()
best_steps_margin['second_best_accuracy'] = second_best['accuracy_mean'].values
best_steps_margin['worst_accuracy'] = worst['accuracy_mean'].values
best_steps_margin['second_best_steps_before_pruning'] = second_best['steps_before_pruning'].values
best_steps_margin['worst_steps_before_pruning'] = worst['steps_before_pruning'].values
best_steps_margin['margin_to_second'] = best_steps_margin['accuracy_mean'] - best_steps_margin['second_best_accuracy']
best_steps_margin['margin_to_worst'] = best_steps_margin['accuracy_mean'] - best_steps_margin['worst_accuracy']
best_steps_margin = best_steps_margin.sort_values(by=['strategy', 'pruning_num_beams', 'branching_factor'])
best_steps_margin[['checkpoint', 'strategy', 'pruning_num_beams', 'branching_factor', 'steps_before_pruning', 'accuracy_mean', 'nfe', 'speed', 'second_best_steps_before_pruning', 'margin_to_second', 'margin_to_worst']]

,checkpoint,strategy,pruning_num_beams,branching_factor,steps_before_pruning,accuracy_mean,nfe,speed,second_best_steps_before_pruning,margin_to_second,margin_to_worst
0,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,1,2,16,0.7172,82.16,0.88,4,0.0094,0.0219
9,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,2,2,4,0.7328,151.46,0.48,8,0.0047,0.0172
14,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,2,4,8,0.7703,235.49,0.32,4,0.0203,0.0484
11,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,4,4,4,0.8125,595.04,0.13,8,0.0219,0.0703
16,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,1,2,16,0.7188,89.49,0.80,4,0.0079,0.0219
25,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,2,2,4,0.7422,168.96,0.43,8,0.0063,0.0188
30,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,2,4,8,0.7781,267.47,0.27,4,0.0203,0.0547
27,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,4,4,4,0.8234,641.33,0.11,8,0.0156,0.0812
32,checkpoints/gidd_0_2/300_epochs,gidd_prob_to_recover_data,1,2,16,0.7156,86.12,0.80,8,0.0047,0.0156
41,checkpoints/gidd_0_2/300_epochs,gidd_prob_to_recover_data,2,2,4,0.7359,158.71,0.44,8,0.0015,0.0187
